#### Simple Gen AI App using LangChain

- Steps: Load Data -> Docs -> Divide into chunks -> Vectors -> Vector Embeddings -> Vector Store DB

In [2]:
#let's import all the details
import os
from dotenv import load_dotenv
load_dotenv()

os.environ['GEMINI_API_KEY']=os.getenv('GEMINI_API_KEY')
#langsmith tracking
os.environ['LANGCHAIN_API_KEY']=os.getenv('LANGCHAIN_API_KEY')
os.environ['LANGCHAIN_TRACING_V2']="true"
os.environ['LANGCHAIN_PROJECT']=os.getenv('LANGCHAIN_PROJECT')

In [3]:
#Data ingestion - from web, we need to scrape the data

from langchain_community.document_loaders import WebBaseLoader

c:\Users\kalya\Projects\Py_Prj_D300526\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


In [4]:
loader=WebBaseLoader("https://docs.langchain.com/langsmith/home")
loader

In [5]:
#load this documents
docs=loader.load()
docs

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/home', 'title': 'LangSmith docs - Docs by LangChain', 'language': 'en'}, page_content='LangSmith docs - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyProfile configurationIntegrationsPlansEnterprise featuresLLM GatewayPrivate betaOverviewCustom model providersSpend policiesPII and secrets redactionAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APITerraform providerAudit logsToolsChatCLISkillsSandboxesAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy

In [6]:
#now divide this entire docs into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter=RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)
documents=text_splitter.split_documents(docs)
documents

[Document(metadata={'source': 'https://docs.langchain.com/langsmith/home', 'title': 'LangSmith docs - Docs by LangChain', 'language': 'en'}, page_content='LangSmith docs - Docs by LangChainDocumentation IndexFetch the complete documentation index at: /llms.txtUse this file to discover all available pages before exploring further.Skip to main contentDocs by LangChain home pageLangSmithSearch...⌘KAsk AIGitHubTry LangSmithTry LangSmithSearch...NavigationLangSmith docsGet startedObservabilityEvaluationPrompt engineeringAgent deploymentPlatform setupReferenceOverviewCreate an account and API keyProfile configurationIntegrationsPlansEnterprise featuresLLM GatewayPrivate betaOverviewCustom model providersSpend policiesPII and secrets redactionAccount administrationOverviewWorkspace setupUsers & access controlBilling & usageManage organizations using the APITerraform providerAudit logsToolsChatCLISkillsSandboxesAdditional resourcesData & complianceFAQLangSmith statusLangSmith docsCopy pageCopy

In [16]:
#convert this entire text into vectors
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings=GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

In [10]:
# using these embeddings, we're converting the text into vectors, so for that, we need to store these vectors in some kind of vector database
#Here, we can take simple example of FAISS Vector Database

from langchain_community.vectorstores import FAISS
vectorstoredb=FAISS.from_documents(documents,embeddings)

In [11]:
vectorstoredb

In [13]:
#from this vectorstoredb, we can query

query="Trace requests, evaluate outputs, test prompts, and manage deployments all in one place, with your agent stack."
result=vectorstoredb.similarity_search(query)
result[0].page_content

'\u200bMore ways to build\nFleetDesign and deploy AI agents visually without writing code.Build an agentPrompt engineeringIterate on prompts with built-in versioning and collaboration to ship improvements faster.Test your promptsLangSmith CLIQuery and manage traces, datasets, experiments, and more from the terminal.Use the CLIStudioUse a visual interface to design, test, and refine applications end-to-end.Develop with Studio\n\u200bInfrastructure\nPlatform setupUse LangSmith in managed cloud, in a self-hosted environment, or hybrid to match your infrastructure and compliance needs.Choose how to set up LangSmithSecurity & complianceLangSmith meets the highest standards of data security and privacy with HIPAA, SOC 2 Type 2, and GDPR compliance. Meet with our team to learn more or visit our Trust Center.Visit Trust Center\n\u200bWorkflow\nLangSmith combines observability, evaluation, deployment, and platform setup in one integrated workflow—from local development to production.'

In [14]:
#using llm 
from langchain_google_genai import ChatGoogleGenerativeAI
llm=ChatGoogleGenerativeAI(model='gemini-2.5-flash')
print(llm)

output_version=None profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-03-20', 'last_updated': '2025-06-05', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True} google_api_key=SecretStr('**********') location=None model='gemini-2.5-flash' client=<google.genai.client.Client object at 0x000001F13AE8BD90> default_metadata=() model_kwargs={}


In [15]:
#Retrieval chain, Document chain

from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_template(
    """
    Answer the following question based only on the provided context:
    <context>
    {context}
    </context>
    """    
)

#This document chain will be responsible in providing my prompt template, this specific context, regarding any input that i'm specifically asking
document_chain=create_stuff_documents_chain(llm,prompt)
document_chain

ModuleNotFoundError: No module named 'langchain.chains'

- Here, we're getting ModuleNotFoundError, since this is using langchain older version of 0.x.x, where as we're using the latest version of langchain as 1.x.x
- So, we're getting this error

- So now, we can use latest as: In LangChain 1.x, you typically use LCEL (LangChain Expression Language) and retrievers directly.
- Example: Modern RAG with LangChain 1.x

In [17]:
# So, we can use this latest retriever
retriever = vectorstoredb.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [19]:
# and use the above prompt for latest as below

# from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate

prompt=ChatPromptTemplate.from_template(
    """
    Answer the following question based only on the provided context:
    Context:
    {context}

    Question:
    {question}
    """    
)

In [20]:
# now, create document chain

from langchain_core.output_parsers import StrOutputParser

document_chain = (
    prompt
    | llm
    | StrOutputParser()
)

document_chain

ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\n    Answer the following question based only on the provided context:\n    Context:\n    {context}\n\n    Question:\n    {question}\n    '), additional_kwargs={})])
| ChatGoogleGenerativeAI(output_version=None, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-03-20', 'last_updated': '2025-06-05', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'image_tool_message':

In [ ]:
# now, test the created document chain

from langchain_core.documents import Document

response = document_chain.invoke({
    "context": [Document(page_content="Generative AI creates new content such as text, images, and code. LangSmith is a framework-agnostic platform for building, debugging, and deploying AI agents and LLM applications. Trace requests, evaluate outputs, test prompts, and manage deployments all in one place, with your agent stack. ")],
    "question": "What is Langsmith?"
})

print(response)

LangSmith is a framework-agnostic platform for building, debugging, and deploying AI agents and LLM applications. It allows users to trace requests, evaluate outputs, test prompts, and manage deployments all in one place, with their agent stack.


However, we want the documents to first come from the retriever we just set up. That way, we can use the retriever to dynamically select the most relevant documents and pass those in for a given question.

In [23]:
### Retriever
'''
    - Retriever can be considered as an Interface
    - It is responsible as if anybody asks any input, then this interface will just be a way of probably getting the data from vector store db.
    - Here, we don't need to do similarity search
    - Here, after creating vector store db, we're converting the vectorstoredb into retriever
    - Input ---> Retriever ---> vectorstoredb
'''

vectorstoredb

In [27]:
# so, inorder to create retriever, we can do the below

from langchain.chains import create_retrieval_chain

retriever=vectorstoredb.as_retriever()
retrieval_chain=create_retrieval_chain(retriever,document_chain)

ModuleNotFoundError: No module named 'langchain.chains'

Since, this is older version, it is showing no module, so we can use the latest version as LECL

In [28]:
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

retriever = vectorstoredb.as_retriever()

retrieval_chain = (
    {
        "context": retriever,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [29]:
retrieval_chain

{
  context: VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x000001F13AF2A7B0>, search_kwargs={}),
  question: RunnablePassthrough()
}
| ChatPromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='\n    Answer the following question based only on the provided context:\n    Context:\n    {context}\n\n    Question:\n    {question}\n    '), additional_kwargs={})])
| ChatGoogleGenerativeAI(output_version=None, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-03-20', 'last_updated': '2025-06-05', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_

In [35]:
# get the response from LLM

response=retrieval_chain.invoke("What is Langsmith?")
response

'LangSmith is a framework-agnostic platform for building, debugging, and deploying AI agents and LLM applications. It allows users to trace requests, evaluate outputs, test prompts, and manage deployments all in one place with their agent stack.'

In [34]:
# just for testing and debugging the details from docs

docs = retriever.invoke("What is Generative AI?")

for doc in docs:
    print(doc.page_content)
    print("-" * 50)

​More ways to build
FleetDesign and deploy AI agents visually without writing code.Build an agentPrompt engineeringIterate on prompts with built-in versioning and collaboration to ship improvements faster.Test your promptsLangSmith CLIQuery and manage traces, datasets, experiments, and more from the terminal.Use the CLIStudioUse a visual interface to design, test, and refine applications end-to-end.Develop with Studio
​Infrastructure
Platform setupUse LangSmith in managed cloud, in a self-hosted environment, or hybrid to match your infrastructure and compliance needs.Choose how to set up LangSmithSecurity & complianceLangSmith meets the highest standards of data security and privacy with HIPAA, SOC 2 Type 2, and GDPR compliance. Meet with our team to learn more or visit our Trust Center.Visit Trust Center
​Workflow
LangSmith combines observability, evaluation, deployment, and platform setup in one integrated workflow—from local development to production.
-------------------------------

In [36]:
response

'LangSmith is a framework-agnostic platform for building, debugging, and deploying AI agents and LLM applications. It allows users to trace requests, evaluate outputs, test prompts, and manage deployments all in one place with their agent stack.'